### `create_react_agent`를 사용하는 이유

#### 1. 정의
- **`create_react_agent`**는  
  LangChain에서 제공하는 **React(Reason + Act) 에이전트**를  
  손쉽게 생성할 수 있는 함수입니다.
- LLM + 여러 도구(tool) + 프롬프트를 결합해  
  **자동으로 멀티스텝 추론과 도구 실행이 반복되는 에이전트 구조**를 만들어줍니다.

#### 2. 사용하는 주요 이유
##### 2.1 LLM의 능동적 문제 해결 프레임워크 제공
- LLM이 단순 응답을 넘어서  
  **스스로 "생각→도구 실행→관찰→최종 답"의 루프를 반복**하도록 설계

##### 2.2 도구와 LLM을 유연하게 연결
- **여러 개의 외부 도구(계산기, 검색, API 등)**를  
  LLM이 필요에 따라 직접 선택·입력값 생성·호출·결과 해석을 자동으로 수행

##### 2.3 표준화된 멀티스텝 추론 패턴 지원
- "Thought-Action-Observation" 패턴을  
  프롬프트에 자동 삽입·누적  
  → 멀티스텝 reasoning/workflow가 쉬워짐

##### 2.4 프롬프트·도구 조합 자동화
- 별도 복잡한 체인 조립 없이  
  LLM, 도구, 프롬프트만 넘기면  
  **최적화된 에이전트가 즉시 완성**

##### 2.5 디버깅/투명성/확장성
- reasoning 과정, 도구 사용 과정이 verbose 모드에서 모두 출력  
- 실시간 로그 추적, 디버깅, 분석이 용이  
- 도구 추가·변경·조합이 유연

#### 3. 실무적 장점
- **AI Copilot, 자동화 챗봇, 분석·요약·검색 등  
  복잡한 워크플로우를 LLM+도구 조합만으로 빠르게 구현** 가능
- 도구가 늘어나도 일관된 인터페이스 유지
- 최신 OpenAI Function Call/Tool Calling과도 구조적으로 유사

#### 4. 한 줄 요약
> **`create_react_agent`는  
> LLM이 멀티스텝 추론을 하며  
> 외부 도구를 능동적으로 사용해 문제를 자동 해결할 수 있도록  
> “최적화된 에이전트 아키텍처”를 코드 한 줄로 빠르게 구현하는 LangChain의 핵심 함수입니다.**

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor, tool

# 환경변수 로드
load_dotenv()

# Gemini LLM 초기화
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)


## 🔧 코드의 목적
AI 챗봇이 수학 계산을 할 수 있도록 "계산기"라는 도구를 만들어 주는 것입니다.

## 📝 각 부분 설명

### 1. `@tool("calculator", return_direct=True)`
```python
@tool("calculator", return_direct=True)
```
- `@tool`: "이것은 AI가 사용할 수 있는 도구야!" 라고 LangChain에게 알려주는 표시
- `"calculator"`: 이 도구의 이름 (AI가 "계산기를 써야겠다" 할 때 찾는 이름)
- `return_direct=True`: "계산 결과를 바로 사용자에게 보여줘!" 라는 의미

### 2. 함수 정의
```python
def calculator(query: str) -> str:
```
- `query: str`: 사용자가 입력한 수학 식 (문자열 형태)
- `-> str`: 결과도 문자열로 돌려준다는 의미

### 3. 설명 문자열
```python
"""수학 계산을 수행합니다. 입력: 수학 표현식 (예: '2+2', '10*5')"""
```
- AI에게 "이 도구가 뭘 하는지, 어떻게 사용하는지" 알려주는 설명서
- AI가 이 설명을 보고 언제 이 도구를 사용할지 판단합니다

### 4. 실제 계산 로직
```python
try:
    result = eval(query)  # 수학 식을 계산
    return f"결과: {result}"
except:
    return "계산 오류가 발생했습니다."  # 오류 처리
```

## 🎯 실제 사용 예시
**사용자**: "25 곱하기 4는 얼마야?"
**AI**: calculator 도구를 사용해서 "25*4"를 계산
**결과**: "결과: 100"

## ⚠️ 주의사항
`eval()` 함수는 보안상 위험할 수 있으므로, 실제 프로덕션에서는 더 안전한 수학 라이브러리(예: `sympy`, `ast.literal_eval`)를 사용하는 것이 좋습니다.
이렇게 만든 도구를 AI 에이전트에 연결하면, AI가 수학 문제를 만났을 때 자동으로 이 계산기를 사용하게 됩니다!

In [ ]:
# 도구 정의 (LangChain의 @tool 데코레이터 사용)
@tool("calculator", return_direct=True)
def calculator(query: str) -> str:
    """수학 계산을 수행합니다. 입력: 수학 표현식 (예: '2+2', '10*5')"""
    try:
        result = eval(query)
        return f"결과: {result}"
    except:
        return "계산 오류가 발생했습니다."

tools = [calculator]

#### 1. ReAct 동작 원리 및 구조
- **Reasoning(추론) + Acting(행동) = AI가 "생각하고 → 행동하고 → 관찰하고 → 다시 생각하는" 방식으로 문제를 해결하는 패턴입니다.**
- **ReAct 프롬프트 템플릿 패턴**
    - Question: 사용자 입력
    - Thought: 다음 행동(혹은 의사결정) 전, 스스로 생각을 표현
    - Action: 사용할 도구 지정(예: 검색, 계산, 외부 API 등)
    - Action Input: 도구에 실제로 입력할 데이터
    - Observation: 도구 실행 결과를 받아 적음
    - (반복)
    - Thought: (필요하면 추가 행동 반복)
    - Final Answer: 최종 답변
##### 2. 변수 설명
- `{tools}`: LLM에게 사용할 도구를 지정
- `{input}`: 사용자가 입력한 질문
- `{agent_scratchpad}`: AI의 이전 생각과 행동들이 누적되는 공간**

- LLM이 이 구조대로 단계별로 응답을 생성하며,매 Thought/Action/Observation 루프마다 실제 도구가 자동 실행되고 그 결과가 다시 LLM의 다음 Thought에 반영됨

In [ ]:


prompt = PromptTemplate.from_template("""
당신은 다음 도구를 사용하여 사용자의 질문에 답합니다:

{tools}

사용 가능한 도구 목록: {tool_names}

질문에 답할 때는 아래의 형식을 반드시 지켜주세요:

Question: 사용자의 질문
Thought: 이 질문을 해결하기 위해 해야할 일을 생각합니다.
Action: [도구 이름] (도구 이름은 반드시 {tool_names} 중 하나여야 합니다!)
Action Input: 도구에 전달할 입력 값
Observation: 도구 사용 결과
...(Thought/Action/Action Input/Observation 반복 가능)
Thought: 이제 최종 답을 알았습니다.
Final Answer: 최종적으로 사용자가 볼 답변

질문: {input}
Thought: {agent_scratchpad}
""")



#### 1. AgentExecutor 역할
- 두뇌 역할: 에이전트가 언제 도구를 사용할지 결정
- 실행 관리: 도구 호출과 결과 처리를 담당
- 안전장치: 무한루프 방지, 오류 처리 등
- verbose=True의 효과 : 실행 과정을 단계별로 보여줘서 AI가 어떻게 생각하고 행동하는지 관찰할 수 있습니다

In [ ]:
# React Agent 생성
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# 에이전트 실행기(Executor) 생성
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 에이전트에게 실제 질문해보기
result = agent_executor.invoke({"input": "23 * 7은 얼마인가요?"})
print("\n=== 에이전트 답변 ===")
print(result["output"])

result2 = agent_executor.invoke({"input": "100 + 2345는?"})
print("\n=== 두 번째 에이전트 답변 ===")
print(result2["output"])